# 02 · Acquisition trend & seasonality

New customers by year, the ~58/week baseline, and the weekly seasonal pattern.

Supports deck slides **2, 3, 4** and the seasonal indices used in the forecast (**8, A4**).

![New customers by year](../reports/figures/s3_new_customers.png)

In [1]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
from common import load, customers, weekly_new, CHANNEL_ORDER, LAST_WEEK_IDX
pd.set_option("display.width", 140)
df = load()

In [2]:
first = df[df.is_first_delivery == 1]
by_year = first.groupby("a_year").size()
print("New customers by acquisition year:", by_year.to_dict())
print("YoY %:", (by_year.pct_change() * 100).round(1).dropna().to_dict())
print("Average new customers per week:", (by_year / 52).round(1).to_dict())
print(f"3-year grand weekly mean: {weekly_new(df).mean():.2f}")

New customers by acquisition year: {2023: 2310, 2024: 3810, 2025: 2942}
YoY %: {2024: 64.9, 2025: -22.8}
Average new customers per week: {2023: 44.4, 2024: 73.3, 2025: 56.6}
3-year grand weekly mean: 58.09


2023 starts at ~24 new customers/week and ramps up, so it is treated as a launch/ramp year: it inflates every 2024 YoY comparison. 2024 → 2025 (−23%) is the cleaner read, and 2025's ~57/week is back near the 3-year baseline of ~58/week.

## Seasonal index

For each ISO week: the 3-year mean of new customers in that week ÷ the grand weekly mean. The 52 indices average to exactly 1.

![Weekly seasonality](../reports/figures/s4_seasonality.png)

In [3]:
w = weekly_new(df).values
iso = np.tile(np.arange(1, 53), 3)
grand = w.mean()
idx = {k: w[iso == k].mean() / grand for k in range(1, 53)}
print(f"grand mean {grand:.2f}; mean of the 52 indices {np.mean(list(idx.values())):.4f}")
print("January W1–W4 indices:", {k: round(float(idx[k]), 2) for k in range(1, 5)})
pk, tr = max(idx, key=idx.get), min(idx, key=idx.get)
print(f"Peak   ISO week {pk}: index {idx[pk]:.2f}  ({w[iso == pk].mean():.0f} new customers/wk)")
print(f"Trough ISO week {tr}: index {idx[tr]:.2f}  ({w[iso == tr].mean():.0f} new customers/wk)")

grand mean 58.09; mean of the 52 indices 1.0000
January W1–W4 indices: {1: 1.21, 2: 1.18, 3: 1.05, 4: 1.11}
Peak   ISO week 36: index 1.57  (91 new customers/wk)
Trough ISO week 52: index 0.60  (35 new customers/wk)


In [4]:
season = pd.DataFrame({"iso_week": range(1, 53), "index": [round(idx[k], 2) for k in range(1, 53)]})
season.sort_values("index", ascending=False).head(8)

,iso_week,index
35,36,1.57
34,35,1.55
36,37,1.54
37,38,1.38
32,33,1.37
38,39,1.36
33,34,1.33
31,32,1.23
